# 1. Dataset sintético de RecoIA

Para entrenar el recomendador necesitamos historial de comportamiento de usuarios. Como no tenemos datos reales de una tienda, generamos un dataset **sintético** que imita un clickstream de e-commerce, siguiendo las reglas definidas en el plan de pruebas de EP2:

- Cada interacción tiene `user_id`, `item_id` y `timestamp` (campos clave, sin nulos).
- Los clics valen 1 y las compras valen 5 (interacciones implícitas positivas).
- Cada usuario tiene una categoría favorita, para que el modelo tenga un patrón real que aprender. Sin patrón, cualquier recomendación sería azar.
- Hay usuarios de tres regiones, con más peso en la zona central, para poder revisar sesgo geográfico después.

Es como inventar clientes de ficción con gustos definidos: los datos son falsos, pero el comportamiento es coherente, y eso permite medir si el modelo realmente aprende.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)  # semilla fija: mismos datos cada vez

N_USERS, N_ITEMS = 1000, 200
categorias = ["Electrónica", "Hogar", "Moda", "Deportes", "Belleza"]

# Catálogo de productos
catalogo = pd.DataFrame({
    "item_id": np.arange(N_ITEMS),
    "categoria": rng.choice(categorias, N_ITEMS),
})

# Usuarios con región y categoría favorita
usuarios = pd.DataFrame({
    "user_id": np.arange(N_USERS),
    "region": rng.choice(["Central", "Norte", "Sur"], N_USERS, p=[0.6, 0.2, 0.2]),
    "cat_favorita": rng.choice(categorias, N_USERS),
})

cat_items = catalogo["categoria"].values
inicio = pd.Timestamp("2026-01-01")
filas = []

for u in usuarios.itertuples():
    n = rng.integers(5, 40)  # interacciones por usuario
    pesos = np.where(cat_items == u.cat_favorita, 5.0, 1.0)
    pesos = pesos / pesos.sum()
    elegidos = rng.choice(N_ITEMS, size=n, p=pesos)
    for it in elegidos:
        p_compra = 0.25 if cat_items[it] == u.cat_favorita else 0.05
        valor = 5 if rng.random() < p_compra else 1
        ts = inicio + pd.Timedelta(minutes=int(rng.integers(0, 60 * 24 * 300)))
        filas.append((u.user_id, it, valor, ts))

interacciones = pd.DataFrame(filas, columns=["user_id", "item_id", "valor", "timestamp"])
interacciones = interacciones.sort_values("timestamp").reset_index(drop=True)

# Guardar archivos
interacciones.to_csv("interacciones.csv", index=False)
catalogo.to_csv("catalogo.csv", index=False)
usuarios.to_csv("usuarios.csv", index=False)

print("Interacciones:", interacciones.shape)
print(interacciones["valor"].value_counts())
print("Nulos:", interacciones.isnull().sum().sum())
interacciones.head()

Interacciones: (22776, 4)
valor
1    19100
5     3676
Name: count, dtype: int64
Nulos: 0


,user_id,item_id,valor,timestamp
0,12,102,1,2026-01-01 00:00:00
1,154,15,5,2026-01-01 00:05:00
2,852,157,1,2026-01-01 00:26:00
3,314,132,1,2026-01-01 00:46:00
4,486,196,1,2026-01-01 00:57:00


### Interpretación del dataset

Se generaron **22.776 interacciones** de 1.000 usuarios sobre un catálogo de 200 productos, es decir, unas 23 interacciones por usuario. No hay valores nulos, por lo que se cumple la regla de integridad definida en el plan de pruebas (sin nulos en `user_id`, `item_id` y `timestamp`).

Del total, 19.100 interacciones son clics (valor 1, un 83,9%) y 3.676 son compras (valor 5, un 16,1%). Esa proporción es coherente con un e-commerce, donde se mira mucho más de lo que se compra.

Con 1.000 usuarios y 200 productos existen 200.000 combinaciones posibles, y solo se observan como máximo unas 22.800. Esto significa que más del 88% de la matriz usuario-producto está vacía. Esa característica, llamada *dispersión*, es normal en los sistemas de recomendación y es justamente lo que el modelo debe aprender a completar.

Las marcas de tiempo avanzan desde el 1 de enero de 2026, lo que nos permite separar el pasado del futuro en el siguiente paso.

# 2. Separación temporal en entrenamiento y prueba

Para evitar *data leakage* (fuga de datos), no mezclamos las interacciones al azar. Ordenamos todo por fecha y usamos el **80% más antiguo para entrenar** y el **20% más reciente para probar**. Así el modelo nunca ve el futuro durante el entrenamiento, igual que un sistema real en producción, que solo conoce lo que ya pasó.

Después verificamos con código que la última fecha de entrenamiento sea anterior a la primera fecha de prueba.

In [2]:
corte = int(len(interacciones) * 0.8)
train = interacciones.iloc[:corte].copy()
test = interacciones.iloc[corte:].copy()

print("Train:", train.shape, "| desde", train["timestamp"].min(), "hasta", train["timestamp"].max())
print("Test: ", test.shape, "| desde", test["timestamp"].min(), "hasta", test["timestamp"].max())

# Verificación anti-fuga: todo el test es posterior al train
assert train["timestamp"].max() <= test["timestamp"].min(), "¡Hay fuga de datos!"
print("OK: el test es estrictamente posterior al train")

train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)

Train: (18220, 4) | desde 2026-01-01 00:00:00 hasta 2026-08-28 07:08:00
Test:  (4556, 4) | desde 2026-08-28 07:09:00 hasta 2026-10-27 23:50:00
OK: el test es estrictamente posterior al train


### Interpretación de la separación temporal

El conjunto de entrenamiento tiene **18.220 interacciones** (80%) y cubre desde el 1 de enero hasta el 28 de agosto de 2026, unos 8 meses. El conjunto de prueba tiene **4.556 interacciones** (20%) y cubre desde el 28 de agosto hasta el 27 de octubre, cerca de 2 meses.

La verificación automática (`assert`) confirmó que la primera interacción de prueba ocurre un minuto después de la última de entrenamiento, por lo que no existe solapamiento temporal. Esto cumple el criterio de prevención de fuga de datos del plan de pruebas: el modelo se evalúa únicamente con comportamiento posterior al que usó para aprender, igual que ocurriría en producción.

# 3. Entrenamiento del modelo de recomendación (ALS)

El modelo elegido es **ALS (Alternating Least Squares)**, un algoritmo de filtrado colaborativo para datos implícitos (clics y compras). Aprende un vector de características latentes para cada usuario y cada producto, de modo que el producto de ambos vectores estima cuánto le gustaría un producto a un usuario.

Primero instalamos la librería `implicit`, que contiene una implementación eficiente de ALS que corre en CPU.

In [4]:
import implicit
from implicit.als import AlternatingLeastSquares
print("Librería cargada correctamente")
print("Versión:", getattr(implicit, "__version__", "no disponible"))

Librería cargada correctamente
Versión: 0.7.3


# 4. Entrenamiento del modelo ALS

Antes de entrenar hay que convertir el historial en una **matriz usuario-producto**: cada fila es un usuario, cada columna es un producto, y cada celda guarda el valor de la interacción (1 por clic, 5 por compra). Como casi toda la matriz está vacía, se guarda en formato disperso (*sparse*), que solo almacena las celdas con datos.

Como en datos implícitos un clic no es una calificación sino una señal de interés, multiplicamos los valores por un factor de confianza (`ALPHA`), de modo que una compra pese mucho más que un clic suelto.

**Hiperparámetros elegidos** (se documentan porque el informe de EP3 los pide):
- `factores = 64`: cuántos "tipos de gusto" ocultos aprende el modelo.
- `regularizacion = 0.05`: evita que el modelo memorice en vez de generalizar.
- `iteraciones = 20`: cuántas pasadas de ajuste hace.
- `ALPHA = 20`: peso de la confianza sobre las interacciones.
- `random_state = 42`: resultados reproducibles.

In [5]:
import numpy as np
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

# Hiperparámetros
ALPHA = 20
FACTORES, REGULARIZACION, ITERACIONES = 64, 0.05, 20

# Matriz usuario-producto (solo con datos de entrenamiento)
user_items = csr_matrix(
    ((train["valor"].values * ALPHA).astype("float32"),
     (train["user_id"].values, train["item_id"].values)),
    shape=(N_USERS, N_ITEMS),
)
dispersion = 1 - user_items.nnz / (N_USERS * N_ITEMS)
print("Matriz:", user_items.shape, "| celdas vacías: {:.1%}".format(dispersion))

# Entrenamiento
modelo = AlternatingLeastSquares(
    factors=FACTORES, regularization=REGULARIZACION,
    iterations=ITERACIONES, random_state=42,
)
modelo.fit(user_items)
print("Factores de usuarios:", modelo.user_factors.shape)
print("Factores de productos:", modelo.item_factors.shape)

# Prueba rápida: recomendaciones para el usuario 0
uid = 0
ids, scores = modelo.recommend(uid, user_items[uid], N=10)
cats = catalogo.set_index("item_id").loc[ids, "categoria"].values
print("\nCategoría favorita del usuario:", usuarios.loc[uid, "cat_favorita"])
for i, c, s in zip(ids, cats, scores):
    print(f"  producto {i:>3} | {c:<12} | puntaje {s:.3f}")

# Guardar el modelo para usarlo después
np.savez("modelo_als.npz",
         user_factors=modelo.user_factors,
         item_factors=modelo.item_factors)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Matriz: (1000, 200) | celdas vacías: 91.7%


100%|██████████| 20/20 [00:00<00:00, 63.61it/s]


Factores de usuarios: (1000, 64)
Factores de productos: (200, 64)

Categoría favorita del usuario: Hogar
  producto  82 | Electrónica  | puntaje 1.070
  producto 149 | Electrónica  | puntaje 0.869
  producto 192 | Deportes     | puntaje 0.849
  producto 141 | Hogar        | puntaje 0.832
  producto 176 | Deportes     | puntaje 0.708
  producto  76 | Hogar        | puntaje 0.662
  producto 145 | Belleza      | puntaje 0.654
  producto  22 | Electrónica  | puntaje 0.641
  producto 105 | Deportes     | puntaje 0.640
  producto 186 | Electrónica  | puntaje 0.606


### Verificación: ¿el modelo aprendió la categoría favorita?

Una prueba con un solo usuario no es concluyente. Para evaluarlo mejor, pedimos las 10 recomendaciones de **cada uno de los 1.000 usuarios** y calculamos qué porcentaje pertenece a su categoría favorita. Lo comparamos con una referencia: la proporción de productos de esa categoría en el catálogo, que sería el resultado esperable si recomendáramos al azar.

In [6]:
# Interacciones de entrenamiento del usuario 0
print("Interacciones de entrenamiento del usuario 0:", (train["user_id"] == 0).sum())

cat_por_item = catalogo.set_index("item_id")["categoria"]
fav = usuarios.set_index("user_id")["cat_favorita"]

# Top-10 para todos los usuarios de una vez
ids_all, _ = modelo.recommend(np.arange(N_USERS), user_items, N=10)

# % del top-10 que cae en la categoría favorita de cada usuario
aciertos = [(cat_por_item.loc[ids_all[u]].values == fav[u]).mean() for u in range(N_USERS)]
azar = [(cat_por_item.values == fav[u]).mean() for u in range(N_USERS)]

print("Recomendaciones en la categoría favorita: {:.1%}".format(np.mean(aciertos)))
print("Referencia al azar (aprox.):               {:.1%}".format(np.mean(azar)))

Interacciones de entrenamiento del usuario 0: 31
Recomendaciones en la categoría favorita: 34.6%
Referencia al azar (aprox.):               19.9%


### Interpretación de la verificación

Al pedir las 10 recomendaciones de los 1.000 usuarios, el **34,6%** pertenece a la categoría favorita del usuario, frente a un **19,9%** esperable si se recomendara al azar. El modelo supera la referencia en aproximadamente 1,7 veces, lo que indica que **sí capturó parte del patrón de preferencias**.

La mejora es moderada y es esperable que sea así: el modelo no recomienda productos que el usuario ya vio (el usuario 0, por ejemplo, tenía 31 interacciones de entrenamiento), por lo que parte de los mejores productos de su categoría favorita queda excluida de la lista. La métrica definitiva de calidad se mide en la siguiente sección con el conjunto de prueba, usando Precision@10, Recall@10 y NDCG@10.

# 5. Evaluación del modelo con el conjunto de prueba

Para cada usuario, le pedimos al modelo 10 recomendaciones usando solo su historial de entrenamiento, y las comparamos con lo que realmente interactuó en el período de prueba (excluyendo productos que ya había visto antes, porque esos nunca se le recomiendan). Se calculan tres métricas de ranking:

- **Precision@10:** de los 10 productos recomendados, qué fracción interesó realmente al usuario.
- **Recall@10:** de todos los productos que le interesaron, qué fracción apareció en la lista.
- **NDCG@10:** igual que la precisión, pero premia que los aciertos aparezcan más arriba en la lista.

Como referencia se usa una **línea base de popularidad**, que recomienda a todos los usuarios los 10 productos más populares que aún no han visto.

In [7]:
K = 10
test_items = test.groupby("user_id")["item_id"].apply(set)
train_items = train.groupby("user_id")["item_id"].apply(set)

def evaluar(recs_fn):
    P, R, N = [], [], []
    for u, rel in test_items.items():
        rel = rel - train_items.get(u, set())   # solo productos nuevos para el usuario
        if not rel:
            continue
        hits = np.array([1 if i in rel else 0 for i in recs_fn(u)])
        P.append(hits.sum() / K)
        R.append(hits.sum() / len(rel))
        dcg = (hits / np.log2(np.arange(2, K + 2))).sum()
        idcg = (1 / np.log2(np.arange(2, min(len(rel), K) + 2))).sum()
        N.append(dcg / idcg)
    return np.mean(P), np.mean(R), np.mean(N), len(P)

# Modelo ALS
res_als = evaluar(lambda u: ids_all[u])

# Línea base: los más populares que el usuario no ha visto
pop_order = train["item_id"].value_counts().index.values
def pop_recs(u):
    vistos = train_items.get(u, set())
    return [i for i in pop_order if i not in vistos][:K]
res_pop = evaluar(pop_recs)

print(f"Usuarios evaluados: {res_als[3]}")
print(f"{'Modelo':<14}{'Precision@10':>14}{'Recall@10':>12}{'NDCG@10':>10}")
print(f"{'ALS':<14}{res_als[0]:>14.4f}{res_als[1]:>12.4f}{res_als[2]:>10.4f}")
print(f"{'Popularidad':<14}{res_pop[0]:>14.4f}{res_pop[1]:>12.4f}{res_pop[2]:>10.4f}")

Usuarios evaluados: 954
Modelo          Precision@10   Recall@10   NDCG@10
ALS                   0.0281      0.0723    0.0526
Popularidad           0.0243      0.0572    0.0397


### Interpretación de la evaluación

Se evaluaron **954 usuarios** que tuvieron al menos un producto nuevo en el período de prueba. El modelo ALS obtuvo Precision@10 = 0,0281, Recall@10 = 0,0723 y NDCG@10 = 0,0526, superando a la línea base de popularidad (0,0243, 0,0572 y 0,0397) en las tres métricas. La mejora relativa es de un 16% en precisión, un 26% en recall y un 33% en NDCG, siendo esta última la más alta, lo que indica que el modelo no solo acierta más, sino que ubica los aciertos en posiciones más altas de la lista.

Los valores absolutos son bajos, algo esperable en la evaluación offline de sistemas de recomendación: el catálogo es pequeño, el período de prueba corto (unos 2 meses) y solo se consideran productos que el usuario no había visto. Cabe señalar que se trata de datos sintéticos y de una única partición temporal, por lo que estos resultados se interpretan como una línea base (champion) contra la cual deberán compararse los reentrenamientos semanales, tal como se definió en el plan de pruebas.

# 6. F1@10 y registro de métricas

El **F1@10** combina precisión y recall en un solo número (su media armónica), de modo que solo es alto si ambas lo son. Se calcula por usuario y luego se promedia. Guardamos todas las métricas del modelo en un archivo `metricas_champion.json`, que será la línea base de referencia para los reentrenamientos.

In [8]:
import json

def f1_medio(recs_fn):
    F = []
    for u, rel in test_items.items():
        rel = rel - train_items.get(u, set())
        if not rel:
            continue
        h = sum(1 for i in recs_fn(u) if i in rel)
        p, r = h / K, h / len(rel)
        F.append(0 if h == 0 else 2 * p * r / (p + r))
    return float(np.mean(F))

f1_als = f1_medio(lambda u: ids_all[u])
f1_pop = f1_medio(pop_recs)
print(f"F1@10  ALS: {f1_als:.4f} | Popularidad: {f1_pop:.4f}")

metricas = {
    "modelo": "ALS (implicit 0.7.3)",
    "hiperparametros": {"factores": FACTORES, "regularizacion": REGULARIZACION,
                        "iteraciones": ITERACIONES, "alpha": ALPHA},
    "usuarios_evaluados": int(res_als[3]),
    "ALS": {"precision@10": float(res_als[0]), "recall@10": float(res_als[1]),
            "ndcg@10": float(res_als[2]), "f1@10": f1_als},
    "popularidad": {"precision@10": float(res_pop[0]), "recall@10": float(res_pop[1]),
                    "ndcg@10": float(res_pop[2]), "f1@10": f1_pop},
}
with open("metricas_champion.json", "w") as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)
print("Métricas guardadas en metricas_champion.json")

F1@10  ALS: 0.0381 | Popularidad: 0.0325
Métricas guardadas en metricas_champion.json


### Interpretación del F1@10

El modelo ALS obtuvo un F1@10 de **0,0381**, frente a 0,0325 de la línea base de popularidad, una mejora relativa de aproximadamente un 17%. Como el F1 solo es alto cuando precisión y recall lo son a la vez, este resultado confirma que la ventaja de ALS no se debe a una sola métrica.

Todas las métricas (Precision@10, Recall@10, NDCG@10 y F1@10) se guardaron en `metricas_champion.json` junto con los hiperparámetros. Ese archivo es la **línea base del modelo champion**: desde ahora, cada reentrenamiento semanal (challenger) debe igualarla o superarla para pasar a producción, tal como se definió en la sección 2.3 del plan de pruebas.

# 7. Empaquetado del modelo para SageMaker

Para desplegar el modelo en un endpoint de SageMaker se necesitan dos cosas:

1. **Los artefactos del modelo** (`model.tar.gz`): los factores aprendidos por ALS, la matriz de productos ya vistos por cada usuario (que se excluyen de las recomendaciones) y el ranking de popularidad (para el fallback de usuarios nuevos).
2. **El código de inferencia** (`inference.py`): define cómo se carga el modelo, cómo se interpreta la solicitud JSON y cómo se calculan las recomendaciones.

Antes de subir nada a la nube, el resultado del código de inferencia se compara con el del modelo original, para asegurar que ambos recomiendan exactamente lo mismo.

In [9]:
import os
os.makedirs("modelo", exist_ok=True)
os.makedirs("codigo", exist_ok=True)

# Factores del modelo (ya guardados antes, los copiamos a la carpeta de artefactos)
np.savez("modelo/modelo_als.npz",
         user_factors=modelo.user_factors,
         item_factors=modelo.item_factors)

# Productos ya vistos por cada usuario + ranking de popularidad (fallback)
vistos = user_items.toarray() > 0
popular = train["item_id"].value_counts().index.values
np.savez("modelo/extras.npz", vistos=vistos, popular=popular)

print("Artefactos:", os.listdir("modelo"))

Artefactos: ['extras.npz', 'modelo_als.npz']


In [10]:
%%writefile codigo/inference.py
import json
import os
import numpy as np


def model_fn(model_dir):
    d = np.load(os.path.join(model_dir, "modelo_als.npz"))
    e = np.load(os.path.join(model_dir, "extras.npz"))
    return {"U": d["user_factors"], "V": d["item_factors"],
            "vistos": e["vistos"], "popular": e["popular"]}


def input_fn(body, content_type="application/json"):
    if content_type != "application/json":
        raise ValueError("Solo se acepta application/json")
    data = json.loads(body)
    return {"user_id": int(data["user_id"]), "k": int(data.get("k", 10))}


def predict_fn(data, m):
    u, k = data["user_id"], max(1, min(data["k"], 50))
    if 0 <= u < m["U"].shape[0]:
        scores = m["U"][u] @ m["V"].T
        scores[m["vistos"][u]] = -np.inf   # no recomendar lo ya visto
        top = np.argsort(-scores)[:k]
        return {"user_id": u, "fallback": False,
                "recomendaciones": [{"item_id": int(i), "score": float(scores[i])} for i in top]}
    # Usuario desconocido (cold start): más populares
    top = m["popular"][:k]
    return {"user_id": u, "fallback": True,
            "recomendaciones": [{"item_id": int(i), "score": None} for i in top]}


def output_fn(pred, accept="application/json"):
    return json.dumps(pred), "application/json"

Writing codigo/inference.py


In [11]:
import sys, tarfile, importlib
sys.path.insert(0, "codigo")
import inference
importlib.reload(inference)

m = inference.model_fn("modelo")

# 1) El código de inferencia recomienda lo mismo que el modelo original
coinciden = 0
for u in range(N_USERS):
    out = inference.predict_fn({"user_id": u, "k": 10}, m)
    ids_inf = [r["item_id"] for r in out["recomendaciones"]]
    coinciden += int(ids_inf == list(ids_all[u]))
print(f"Usuarios con recomendaciones idénticas: {coinciden}/{N_USERS}")

# 2) Fallback para un usuario que no existe
print("Usuario 99999:", inference.predict_fn({"user_id": 99999, "k": 3}, m))

# 3) Crear model.tar.gz solo con los artefactos
with tarfile.open("model.tar.gz", "w:gz") as tar:
    for f in os.listdir("modelo"):
        tar.add(os.path.join("modelo", f), arcname=f)
print("model.tar.gz:", round(os.path.getsize("model.tar.gz") / 1024, 1), "KB")

Usuarios con recomendaciones idénticas: 1000/1000
Usuario 99999: {'user_id': 99999, 'fallback': True, 'recomendaciones': [{'item_id': 82, 'score': None}, {'item_id': 31, 'score': None}, {'item_id': 43, 'score': None}]}
model.tar.gz: 296.4 KB


### Interpretación de la verificación del empaquetado

El código de inferencia (`inference.py`) produjo recomendaciones **idénticas a las del modelo original en los 1.000 usuarios** (1000/1000), por lo que el empaquetado no altera el comportamiento del modelo. Además, se verificó el **mecanismo de fallback**: ante un usuario inexistente (`user_id` = 99999), el sistema no falla, sino que responde con los productos más populares y marca `fallback: true`, tal como se definió para el *cold start* en el diseño de EP2.

El artefacto final `model.tar.gz` pesa 296 KB, lo que permite desplegarlo en un endpoint serverless con arranque rápido y costo mínimo.

# 8. Subida del modelo a Amazon S3

El endpoint de SageMaker carga el modelo desde un bucket de S3. En esta celda se verifica el entorno (versión del SDK de SageMaker, región, cuenta y rol de ejecución) y se sube `model.tar.gz` a un bucket del proyecto.

In [12]:
import boto3, sagemaker

print("SDK de SageMaker:", sagemaker.__version__)

region = boto3.Session().region_name
account = boto3.client("sts").get_caller_identity()["Account"]
try:
    role = sagemaker.get_execution_role()
except Exception as e:
    role = f"(no se pudo obtener: {e})"
print("Región:", region)
print("Rol:", role)

# Bucket del proyecto (se crea si no existe)
bucket = f"sagemaker-{region}-{account}"
s3 = boto3.client("s3", region_name=region)
try:
    s3.head_bucket(Bucket=bucket)
    print("Bucket ya existe:", bucket)
except Exception:
    s3.create_bucket(Bucket=bucket)
    print("Bucket creado:", bucket)

# Subir el modelo
clave = "recoia/modelo/model.tar.gz"
s3.upload_file("model.tar.gz", bucket, clave)
model_data = f"s3://{bucket}/{clave}"
print("Modelo subido a:", model_data)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


SDK de SageMaker: 2.257.7
Región: us-east-1
Rol: arn:aws:iam::563930113430:role/LabRole
Bucket creado: sagemaker-us-east-1-563930113430
Modelo subido a: s3://sagemaker-us-east-1-563930113430/recoia/modelo/model.tar.gz


### Interpretación de la subida a S3

El entorno se verificó correctamente: región **us-east-1**, SDK de SageMaker 2.257.7 y rol de ejecución **LabRole**, el rol institucional de AWS Academy. Se creó el bucket `sagemaker-us-east-1-563930113430` y el artefacto del modelo quedó disponible en `s3://sagemaker-us-east-1-563930113430/recoia/modelo/model.tar.gz`, desde donde el endpoint lo cargará al iniciarse.

# 9. Despliegue del endpoint serverless en SageMaker

Se crea un **endpoint serverless**: en vez de mantener un servidor encendido todo el día, SageMaker levanta la capacidad solo cuando llega una solicitud y la apaga cuando no hay tráfico. Es como una cocina que se enciende únicamente cuando entra un pedido: se paga por uso y no por tiempo ocioso, lo que encaja con el requisito de escalabilidad y reducción de costos.

Parámetros elegidos: 1.024 MB de memoria y concurrencia máxima de 5 solicitudes simultáneas. Después del despliegue se prueba el endpoint y se mide su latencia, distinguiendo la primera llamada (arranque en frío) de las siguientes.

In [13]:
# Ajuste de output_fn para que devuelva solo el JSON como texto
ruta = "codigo/inference.py"
src = open(ruta).read()
src = src.replace('return json.dumps(pred), "application/json"', 'return json.dumps(pred)')
open(ruta, "w").write(src)

# Reempaquetar y subir de nuevo
with tarfile.open("model.tar.gz", "w:gz") as tar:
    for f in os.listdir("modelo"):
        tar.add(os.path.join("modelo", f), arcname=f)
s3.upload_file("model.tar.gz", bucket, clave)
print("output_fn actualizado y modelo resubido a:", model_data)
print("Revisión:", "return json.dumps(pred)" in open(ruta).read())

output_fn actualizado y modelo resubido a: s3://sagemaker-us-east-1-563930113430/recoia/modelo/model.tar.gz
Revisión: True


In [14]:
import time, json
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.serverless import ServerlessInferenceConfig

sk_model = SKLearnModel(
    model_data=model_data,
    role=role,
    entry_point="inference.py",
    source_dir="codigo",
    framework_version="1.2-1",
    py_version="py3",
)

serverless = ServerlessInferenceConfig(memory_size_in_mb=1024, max_concurrency=5)

predictor = sk_model.deploy(
    serverless_inference_config=serverless,
    endpoint_name="recoia-endpoint",
)
print("\nEndpoint desplegado:", predictor.endpoint_name)

# Prueba con el runtime de SageMaker
runtime = boto3.client("sagemaker-runtime", region_name=region)

def consultar(user_id, k=5):
    t0 = time.time()
    r = runtime.invoke_endpoint(
        EndpointName="recoia-endpoint",
        ContentType="application/json",
        Body=json.dumps({"user_id": user_id, "k": k}),
    )
    out = json.loads(r["Body"].read())
    return out, (time.time() - t0) * 1000

out, ms = consultar(0)
print(f"Primera llamada (arranque en frío): {ms:.0f} ms")
print(out)

tiempos = [consultar(u)[1] for u in range(1, 11)]
print(f"Siguientes 10 llamadas: promedio {np.mean(tiempos):.0f} ms | máximo {np.max(tiempos):.0f} ms")

print("Usuario nuevo:", consultar(99999, 3)[0])

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: SKLearnModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
SKLearnModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


-----!

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/base_predictor.py:140: SageMakerV2DeprecationWarning: SKLearnPredictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
SKLearnPredictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.



Endpoint desplegado: recoia-endpoint
Primera llamada (arranque en frío): 144 ms
{'user_id': 0, 'fallback': False, 'recomendaciones': [{'item_id': 82, 'score': 1.0695276260375977}, {'item_id': 149, 'score': 0.8692827224731445}, {'item_id': 192, 'score': 0.8494078516960144}, {'item_id': 141, 'score': 0.8324282169342041}, {'item_id': 176, 'score': 0.7081475853919983}]}
Siguientes 10 llamadas: promedio 1108 ms | máximo 10664 ms
Usuario nuevo: {'user_id': 99999, 'fallback': True, 'recomendaciones': [{'item_id': 82, 'score': None}, {'item_id': 31, 'score': None}, {'item_id': 43, 'score': None}]}


### Interpretación del despliegue

El endpoint **recoia-endpoint** se desplegó correctamente en modo serverless (1.024 MB de memoria, concurrencia máxima de 5). Las recomendaciones que devuelve para el usuario 0 son idénticas a las del modelo entrenado localmente, por lo que el despliegue no altera el comportamiento del modelo. Para un usuario inexistente (`user_id` = 99999) el endpoint responde con el fallback de popularidad (`fallback: true`) en lugar de fallar.

En la prueba preliminar de latencia, la primera llamada tomó 144 ms. De las 10 llamadas siguientes, la mayoría respondió en decenas de milisegundos, pero una alcanzó 10.664 ms, lo que elevó el promedio a 1.108 ms. Este valor atípico es consistente con un arranque en frío del contenedor serverless; su frecuencia real se analiza en la sección siguiente con una muestra mayor.

# 10. Medición de latencia del endpoint

Se realizan 100 llamadas consecutivas a usuarios distintos y se calculan los percentiles de latencia: **p50** (la mitad de las llamadas es más rápida que este valor), **p95** y **p99** (el peor 5% y el peor 1%). Se compara el resultado con el SLA definido en EP2 (p95 ≤ 200 ms) y se cuenta cuántas llamadas lo superan.

In [ ]:
tiempos = []
for u in range(100):
    _, ms = consultar(u, 10)
    tiempos.append(ms)
t = np.array(tiempos)

print(f"Llamadas: {len(t)}")
print(f"p50: {np.percentile(t, 50):.0f} ms")
print(f"p95: {np.percentile(t, 95):.0f} ms")
print(f"p99: {np.percentile(t, 99):.0f} ms")
print(f"Máximo: {t.max():.0f} ms | Mínimo: {t.min():.0f} ms")
print(f"Llamadas sobre 200 ms: {(t > 200).sum()} de {len(t)}")

### Interpretación de la latencia

Sobre 100 llamadas consecutivas, el endpoint obtuvo una latencia **p50 de 41 ms**, **p95 de 1.097 ms** y **p99 de 1.298 ms** (mínimo 21 ms, máximo 1.536 ms). La mediana cumple con holgura el SLA definido en EP2, pero el **p95 lo supera ampliamente**: 17 de las 100 llamadas excedieron los 200 ms.

La diferencia entre la mediana y los percentiles altos indica que la mayoría de las solicitudes se atiende rápido, mientras que una fracción considerable sufre retrasos cercanos a un segundo. La medición se realizó desde la instancia de notebook, por lo que incluye la red y la invocación del servicio, no solo el cómputo del modelo. En la siguiente sección se analizan las métricas de CloudWatch del endpoint para identificar el origen de esos retrasos.

# 11. Análisis de la latencia con métricas de CloudWatch

SageMaker publica en Amazon CloudWatch métricas propias de cada endpoint serverless. Las relevantes son: **ModelSetupTime** (tiempo de levantar capacidad nueva, es decir, los arranques en frío), **ModelLatency** (tiempo que tarda el modelo en responder) y **OverheadLatency** (tiempo añadido por el servicio). Comparándolas se distingue si los retrasos vienen de arranques en frío o del propio cómputo.

In [16]:
import datetime as dt
cw = boto3.client("cloudwatch", region_name=region)
fin = dt.datetime.utcnow()
ini = fin - dt.timedelta(minutes=60)

def metrica(nombre):
    r = cw.get_metric_statistics(
        Namespace="AWS/SageMaker", MetricName=nombre,
        Dimensions=[{"Name": "EndpointName", "Value": "recoia-endpoint"},
                    {"Name": "VariantName", "Value": "AllTraffic"}],
        StartTime=ini, EndTime=fin, Period=60,
        Statistics=["Average", "Maximum", "SampleCount"],
    )
    return r["Datapoints"]

for nombre in ["Invocations", "ModelSetupTime", "ModelLatency", "OverheadLatency"]:
    pts = metrica(nombre)
    if not pts:
        print(f"{nombre:<16} sin datos todavía")
        continue
    if nombre == "Invocations":
        print(f"{nombre:<16} total: {sum(p['SampleCount'] for p in pts):.0f}")
    else:
        n = sum(p["SampleCount"] for p in pts)
        prom = np.average([p["Average"] for p in pts], weights=[p["SampleCount"] for p in pts]) / 1000
        mx = max(p["Maximum"] for p in pts) / 1000
        print(f"{nombre:<16} eventos: {n:.0f} | promedio: {prom:.0f} ms | máximo: {mx:.0f} ms")

Invocations      total: 112
ModelSetupTime   eventos: 1 | promedio: 10297 ms | máximo: 10297 ms
ModelLatency     eventos: 112 | promedio: 2 ms | máximo: 21 ms
OverheadLatency  eventos: 112 | promedio: 243 ms | máximo: 10542 ms


### Interpretación de las métricas de CloudWatch

Durante la ventana analizada se registraron **112 invocaciones**. El **ModelLatency** fue de 2 ms en promedio (máximo 21 ms), lo que muestra que el cómputo del modelo es despreciable. Se registró un único evento de **ModelSetupTime** de 10.297 ms, correspondiente al arranque en frío del contenedor serverless, y que explica el valor máximo de la prueba preliminar.

La **OverheadLatency**, es decir, el tiempo añadido por el servicio fuera del modelo, promedió 243 ms con un máximo de 10.542 ms (el arranque en frío). Dado que hubo un solo arranque en frío, las llamadas de cerca de un segundo observadas en la medición de 100 llamadas no se explican por arranques en frío, sino por la sobrecarga del servicio. Su causa se investiga en la siguiente sección repitiendo la medición.

# 12. Repetición de la medición de latencia

Para comprobar si el comportamiento observado es reproducible, se realizan 200 llamadas consecutivas y se registra la posición de cada llamada que supera los 200 ms, además de los percentiles por bloques de 50 llamadas. Esto permite distinguir un efecto de calentamiento (llamadas lentas concentradas al inicio) de un comportamiento disperso del servicio.

In [18]:
tiempos2 = []
for i in range(200):
    _, ms = consultar(i % N_USERS, 10)
    tiempos2.append(ms)
t2 = np.array(tiempos2)

print(f"p50: {np.percentile(t2, 50):.0f} ms | p95: {np.percentile(t2, 95):.0f} ms | p99: {np.percentile(t2, 99):.0f} ms")
print(f"Llamadas sobre 200 ms: {(t2 > 200).sum()} de {len(t2)}")
print("Posiciones de las lentas:", np.where(t2 > 200)[0].tolist())

for b in range(0, 200, 50):
    bloque = t2[b:b + 50]
    print(f"Llamadas {b:>3}-{b + 49:<3} | p50: {np.percentile(bloque, 50):>5.0f} ms | p95: {np.percentile(bloque, 95):>5.0f} ms")

p50: 40 ms | p95: 80 ms | p99: 124 ms
Llamadas sobre 200 ms: 0 de 200
Posiciones de las lentas: []
Llamadas   0-49  | p50:    40 ms | p95:    72 ms
Llamadas  50-99  | p50:    40 ms | p95:    81 ms
Llamadas 100-149 | p50:    40 ms | p95:   104 ms
Llamadas 150-199 | p50:    41 ms | p95:    60 ms
